# Senate meeting minutes processor

## Configuration

In [7]:
WORK_DIR = "senate_meeting_minutes"
INPUT_URL = "https://www.uni-corvinus.hu/fooldal/egyetemunkrol/szenatus-egyetemi-testuletek/szenatus/"

## URL crawler

In [ ]:
import re

import requests

import pandas as pd

from pathlib import Path
from getpass import getpass
from bs4 import BeautifulSoup

from urllib.parse import unquote, urljoin, urlparse

In [ ]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/151.0.0.0 Safari/537.36"
    )
}

session = requests.Session()
session.headers.update(headers)

protected_page = session.get(INPUT_URL, timeout=30)
protected_soup = BeautifulSoup(protected_page.text, "html.parser")
login_link = protected_soup.find(
    "a",
    href=re.compile(r"wp-login\.php(?:\?|$)"),
)

if (
    protected_page.status_code < 400
    and protected_soup.select_one(".sidebar-nav") is not None
):
    authenticated_page = protected_page
    authenticated_soup = protected_soup
elif login_link is None:
    protected_page.raise_for_status()
    raise RuntimeError(
        f"Could not find .sidebar-nav or a login link at {protected_page.url!r}."
    )
else:
    login_page = session.get(
        urljoin(protected_page.url, login_link["href"]),
        timeout=30,
    )
    login_soup = BeautifulSoup(login_page.text, "html.parser")
    login_form = login_soup.find("form")

    if login_page.status_code >= 400 and login_page.status_code != 401:
        login_page.raise_for_status()
    if login_form is None:
        if login_page.status_code == 401:
            raise RuntimeError(
                f"The SSO login page returned 401 Unauthorized at {login_page.url!r}; "
                "the login flow cannot continue because no login form was provided."
            )
        raise RuntimeError(f"Could not find the login form at {login_page.url!r}.")

    hidden_fields = {
        field["name"]: field.get("value", "")
        for field in login_form.select("input[type=hidden][name]")
    }

    username = input("Corvinus username: ").strip()
    if not username:
        raise ValueError("A Corvinus username is required.")

    password = getpass("Corvinus password: ")
    if not password:
        raise ValueError("A Corvinus password is required.")

    login_data = {
        **hidden_fields,
        "username": username,
        "password": password,
    }
    login_data.setdefault("_eventId", "submit")

    login_action = urljoin(
        login_page.url,
        login_form.get("action") or login_page.url,
    )
    authenticated_page = session.post(
        login_action,
        data=login_data,
        timeout=30,
    )
    authenticated_soup = BeautifulSoup(authenticated_page.text, "html.parser")

    if authenticated_page.status_code == 401 or authenticated_page.url.startswith(
        "https://sso.uni-corvinus.hu/cas/login"
    ):
        error = authenticated_soup.select_one("#msg, .errors, .alert-danger")
        detail = error.get_text(" ", strip=True) if error else "credentials were rejected"
        raise RuntimeError(f"CAS login failed: {detail}")

    authenticated_page.raise_for_status()

In [ ]:
output_dir = Path.cwd() / WORK_DIR
output_dir.mkdir(parents=True, exist_ok=True)

sidebar_links = []
seen_sidebar_urls = set()
for link in authenticated_soup.select(".sidebar-nav a[href]"):
    sidebar_url = urljoin(authenticated_page.url, link["href"])
    if sidebar_url in seen_sidebar_urls:
        continue

    seen_sidebar_urls.add(sidebar_url)
    sidebar_label = link.get_text(" ", strip=True)
    if not sidebar_label:
        sidebar_label = link.get("aria-label") or link.get("title") or sidebar_url

    sidebar_links.append(
        {
            "label": sidebar_label,
            "url": sidebar_url,
        }
    )

if not sidebar_links:
    raise RuntimeError(
        "No links were found in .sidebar-nav after login; the page may have changed "
        "or the account may not have access."
    )

pd.DataFrame(sidebar_links, columns=["label", "url"]).to_csv(
    output_dir / "senate_sidebar_links.csv",
    index=False,
    encoding="utf-8-sig",
)

In [ ]:
pdf_records = []
not_found_links = []
unauthorized_links = []
connection_error_links = []
seen_pdf_urls = set()
for page_index, sidebar_link in enumerate(sidebar_links, start=1):
    print(
        f"Reading {page_index}/{len(sidebar_links)}: "
        f"{sidebar_link['label']}"
    )
    try:
        section_page = session.get(sidebar_link["url"], timeout=30)
    except requests.exceptions.ConnectionError as error:
        print(f"  Connection error: {sidebar_link['url']} ({error})")
        connection_error_links.append(
            {
                "type": "sidebar_page",
                "label": sidebar_link["label"],
                "url": sidebar_link["url"],
                "error": str(error),
            }
        )
        continue

    if section_page.status_code == 401:
        print(f"  401 Unauthorized: {section_page.url}")
        unauthorized_links.append(
            {
                "type": "sidebar_page",
                "label": sidebar_link["label"],
                "url": sidebar_link["url"],
                "error": "Unauthorized",
            }
        )
        continue

    if section_page.status_code == 404:
        print(f"  404 Not Found: {section_page.url}")
        not_found_links.append(
            {
                "type": "sidebar_page",
                "label": sidebar_link["label"],
                "url": sidebar_link["url"],
            }
        )
        continue

    section_page.raise_for_status()
    section_soup = BeautifulSoup(section_page.text, "html.parser")
    main = section_soup.find("main")

    if main is None:
        print(f"  No <main> tag found at {section_page.url}")
        continue

    for pdf_link in main.select("a[href]"):
        pdf_url = urljoin(section_page.url, pdf_link["href"])
        if not urlparse(pdf_url).path.lower().endswith(".pdf"):
            continue
        if pdf_url in seen_pdf_urls:
            continue

        seen_pdf_urls.add(pdf_url)
        pdf_label = pdf_link.get_text(" ", strip=True)
        if not pdf_label:
            pdf_label = pdf_link.get("aria-label") or pdf_link.get("title") or pdf_url

        pdf_records.append(
            {
                "sidebar_label": sidebar_link["label"],
                "sidebar_url": sidebar_link["url"],
                "pdf_label": pdf_label,
                "pdf_url": pdf_url,
                "status": "pending",
                "error": "",
            }
        )

if (
    not pdf_records
    and not not_found_links
    and not unauthorized_links
    and not connection_error_links
):
    raise RuntimeError(
        "No PDF links were found in the <main> tag of the sidebar pages; "
        "the page structure or account access may have changed."
    )

In [ ]:
used_filenames = set()
for index, record in enumerate(pdf_records, start=1):
    pdf_url = record["pdf_url"]
    filename = Path(unquote(urlparse(pdf_url).path)).name
    filename = re.sub(r'[<>:"/\\|?*\x00-\x1f]', "_", filename)
    if not filename:
        filename = f"senate_{index:04d}.pdf"

    original_filename = filename
    duplicate_number = 2
    while filename in used_filenames:
        path = Path(original_filename)
        filename = f"{path.stem}_{duplicate_number}{path.suffix}"
        duplicate_number += 1
    used_filenames.add(filename)
    record["filename"] = filename

    destination = output_dir / filename
    partial_destination = destination.with_suffix(destination.suffix + ".part")
    print(f"Downloading {index}/{len(pdf_records)}: {filename}")

    try:
        with session.get(pdf_url, stream=True, timeout=60) as pdf_response:
            if pdf_response.status_code == 401:
                record["status"] = "401"
                record["error"] = "Unauthorized"
                unauthorized_links.append(
                    {
                        "type": "pdf",
                        "label": record["pdf_label"],
                        "url": pdf_url,
                        "error": "Unauthorized",
                    }
                )
                print(f"  401 Unauthorized: {pdf_url}")
                continue

            if pdf_response.status_code == 404:
                record["status"] = "404"
                record["error"] = "Not Found"
                not_found_links.append(
                    {
                        "type": "pdf",
                        "label": record["pdf_label"],
                        "url": pdf_url,
                    }
                )
                print(f"  404 Not Found: {pdf_url}")
                continue

            pdf_response.raise_for_status()
            first_chunk = next(pdf_response.iter_content(chunk_size=8192), b"")
            if not first_chunk.startswith(b"%PDF"):
                raise RuntimeError(
                    f"The response for {pdf_url} was not a PDF; "
                    "the authenticated session may have expired."
                )

            with partial_destination.open("wb") as output_file:
                output_file.write(first_chunk)
                for chunk in pdf_response.iter_content(chunk_size=8192):
                    if chunk:
                        output_file.write(chunk)

        partial_destination.replace(destination)
        record["status"] = "downloaded"
    except requests.exceptions.ConnectionError as error:
        record["status"] = "connection_error"
        record["error"] = str(error)
        connection_error_links.append(
            {
                "type": "pdf",
                "label": record["pdf_label"],
                "url": pdf_url,
                "error": str(error),
            }
        )
        print(f"  Connection error: {pdf_url} ({error})")
    finally:
        try:
            partial_destination.unlink()
        except FileNotFoundError:
            pass

In [ ]:
manifest_columns = [
    "sidebar_label",
    "sidebar_url",
    "pdf_label",
    "pdf_url",
    "filename",
    "status",
    "error",
]
pd.DataFrame(pdf_records, columns=manifest_columns).to_csv(
    output_dir / "senate_pdf_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)
pd.DataFrame(not_found_links, columns=["type", "label", "url"]).to_csv(
    output_dir / "senate_404_links.csv",
    index=False,
    encoding="utf-8-sig",
)
pd.DataFrame(
    unauthorized_links,
    columns=["type", "label", "url", "error"],
).to_csv(
    output_dir / "senate_401_links.csv",
    index=False,
    encoding="utf-8-sig",
)
pd.DataFrame(
    connection_error_links,
    columns=["type", "label", "url", "error"],
).to_csv(
    output_dir / "senate_connection_errors.csv",
    index=False,
    encoding="utf-8-sig",
)

downloaded_count = sum(record["status"] == "downloaded" for record in pdf_records)
print(f"Downloaded {downloaded_count} of {len(pdf_records)} PDFs to {output_dir}")
print(f"Saved PDF metadata to {output_dir / 'senate_pdf_manifest.csv'}")
if not_found_links:
    print(
        f"Skipped {len(not_found_links)} links that returned 404; "
        f"details saved to {output_dir / 'senate_404_links.csv'}"
    )
if unauthorized_links:
    print(
        f"Skipped {len(unauthorized_links)} links that returned 401; "
        f"details saved to {output_dir / 'senate_401_links.csv'}"
    )
if connection_error_links:
    print(
        f"Skipped {len(connection_error_links)} links due to connection errors; "
        f"details saved to {output_dir / 'senate_connection_errors.csv'}"
    )

## OCRing the PDFs

In [22]:
import re
import json
from pathlib import Path

import fitz  # PyMuPDF
import pandas as pd

from PIL import Image

In [23]:
OCR_LANGUAGE = "hun"
OCR_DPI = 300

In [24]:
def extract_pdf_text(pdf_path: Path) -> str:
    """
    Extract text from a PDF.

    First tries normal PDF text extraction.
    If the PDF is scanned/image-based, falls back to OCR.
    """

    print(f"Processing: {pdf_path.name}")

    with fitz.open(pdf_path) as doc:

        # -----------------------------------------------------------
        # 1. Try native PDF text extraction
        # -----------------------------------------------------------

        native_text = "\n\n".join(
            page.get_text("text")
            for page in doc
        ).strip()

        # If enough text was extracted, use it
        if len(native_text) > 100:
            print("  -> Native text extraction")
            return native_text

        # -----------------------------------------------------------
        # 2. OCR fallback
        # -----------------------------------------------------------

        print("  -> Scanned PDF detected, using OCR")

        ocr_pages = []

        zoom = OCR_DPI / 72
        matrix = fitz.Matrix(zoom, zoom)

        for page_number, page in enumerate(doc, start=1):

            print(f"     OCR page {page_number}/{len(doc)}")

            # Render page to image
            pix = page.get_pixmap(
                matrix=matrix,
                alpha=False
            )

            image = Image.frombytes(
                "RGB",
                [pix.width, pix.height],
                pix.samples
            )

            # OCR
            text = pytesseract.image_to_string(
                image,
                lang=OCR_LANGUAGE,
                config="--psm 3"
            )

            ocr_pages.append(text.strip())

        return "\n\n".join(ocr_pages)


In [25]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

print(pytesseract.get_languages())

['eng', 'hun', 'osd']


In [26]:
if not Path(WORK_DIR).exists():
    raise FileNotFoundError(
        f"Output directory does not exist: {WORK_DIR}"
    )

pdf_files = sorted(Path(WORK_DIR).glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files\n")


# ---------------------------------------------------------
# Checkpoint configuration
# ---------------------------------------------------------

CHECKPOINT_FILE = Path(WORK_DIR) / "extraction_checkpoint.json"


# Load existing checkpoint if available
if CHECKPOINT_FILE.exists():
    try:
        with CHECKPOINT_FILE.open("r", encoding="utf-8") as f:
            records = json.load(f)

        print(
            f"Loaded checkpoint: {len(records)} PDFs already processed\n"
        )

    except (json.JSONDecodeError, OSError) as e:
        print(f"Could not load checkpoint: {e}")
        print("Starting with an empty checkpoint.\n")
        records = []
else:
    records = []


# Keep track of files already successfully processed
processed_files = {
    record["filename"]
    for record in records
}


# ---------------------------------------------------------
# Process PDFs
# ---------------------------------------------------------

for pdf_path in pdf_files:

    # Skip files already successfully processed
    if pdf_path.name in processed_files:
        print(f"SKIP: {pdf_path.name}")
        continue

    print(f"Processing: {pdf_path.name}")

    try:
        text = extract_pdf_text(pdf_path)

        # Extract year from filename, e.g.
        # 2020_some_document.pdf -> 2020
        year_match = re.match(
            r"^(\d{4})_",
            pdf_path.name
        )

        year = (
            int(year_match.group(1))
            if year_match
            else None
        )

        record = {
            "year": year,
            "filename": pdf_path.name,
            "text": text,
        }

        records.append(record)

        # -------------------------------------------------
        # Save checkpoint immediately after each PDF
        # -------------------------------------------------

        with CHECKPOINT_FILE.open(
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                records,
                f,
                ensure_ascii=False,
                indent=2
            )

        processed_files.add(pdf_path.name)

        print(
            f"  Extracted {len(text):,} characters"
        )
        print(
            f"  Checkpoint saved ({len(records)} PDFs)\n"
        )

    except Exception as e:
        print(
            f"ERROR processing {pdf_path.name}: {e}\n"
        )

Found 445 PDF files

Processing: 1-1-beszamolo-2025.pdf
Processing: 1-1-beszamolo-2025.pdf
  -> Scanned PDF detected, using OCR
     OCR page 1/56
     OCR page 2/56
     OCR page 3/56
     OCR page 4/56
     OCR page 5/56
     OCR page 6/56
     OCR page 7/56
     OCR page 8/56
     OCR page 9/56
     OCR page 10/56
     OCR page 11/56
     OCR page 12/56
     OCR page 13/56
     OCR page 14/56
     OCR page 15/56
     OCR page 16/56
     OCR page 17/56
     OCR page 18/56
     OCR page 19/56
     OCR page 20/56
     OCR page 21/56
     OCR page 22/56
     OCR page 23/56
     OCR page 24/56
     OCR page 25/56
     OCR page 26/56
     OCR page 27/56
     OCR page 28/56
     OCR page 29/56
     OCR page 30/56
     OCR page 31/56
     OCR page 32/56
     OCR page 33/56
     OCR page 34/56
     OCR page 35/56
     OCR page 36/56
     OCR page 37/56
     OCR page 38/56
     OCR page 39/56
     OCR page 40/56
     OCR page 41/56
     OCR page 42/56
     OCR page 43/56
     OCR page 44/56
 

In [27]:
from Helper import Log

In [28]:
df = pd.DataFrame(records)

In [48]:
import statistics

print("Average length of OCRd text: {:,.2f}".format(
    statistics.mean([i for i in df['text'].str.len()])
))

Average length of OCRd text: 60,634.55


In [49]:
with Log.save_file(WORK_DIR + "/output/senate_meeting_minutes_hun.csv") as filename:
    df.to_csv(filename, index=False)

File saved to `senate_meeting_minutes\output\senate_meeting_minutes_hun_202609231622.csv` (28.0MB)


## Optinal chunking

In [5]:
import pandas as pd
from Helper import Log

In [3]:
df = pd.read_csv('senate_meeting_minutes\output\senate_meeting_minutes_hun_202609231622.csv')

In [9]:
import os
import io

MAX_BYTES = 1 * 1024 * 1024  # 1 MB

part_number = 1
start = 0
n = len(df)

while start < n:
    # Grow the chunk until the next row would push us over 1 MB
    end = start + 1
    while end <= n:
        candidate = df.iloc[start:end]
        # Measure the exact size the JSON will have on disk
        buffer = io.StringIO()
        candidate.to_json(
            buffer,
            orient="records",
            force_ascii=False,
            indent=2,
        )
        size = len(buffer.getvalue().encode("utf-8"))
        if size > MAX_BYTES and end > start + 1:
            # Last row made it too big → drop it
            end -= 1
            break
        end += 1

    chunk = df.iloc[start:end]
    with Log.save_file(f"{WORK_DIR}/json/senate_meeting_minutes_hun_part{part_number}.json") as filename:
        chunk.to_json(
            filename,
            orient="records",
            force_ascii=False,
            indent=2,
        )

    start = end
    part_number += 1

File saved to `senate_meeting_minutes\json\senate_meeting_minutes_hun_part1_202609272025.json` (1013.5KB)
File saved to `senate_meeting_minutes\json\senate_meeting_minutes_hun_part2_202609272025.json` (917.1KB)
File saved to `senate_meeting_minutes\json\senate_meeting_minutes_hun_part3_202609272025.json` (996.9KB)
File saved to `senate_meeting_minutes\json\senate_meeting_minutes_hun_part4_202609272025.json` (1006.0KB)
File saved to `senate_meeting_minutes\json\senate_meeting_minutes_hun_part5_202609272025.json` (864.9KB)
File saved to `senate_meeting_minutes\json\senate_meeting_minutes_hun_part6_202609272025.json` (995.7KB)
File saved to `senate_meeting_minutes\json\senate_meeting_minutes_hun_part7_202609272025.json` (924.7KB)
File saved to `senate_meeting_minutes\json\senate_meeting_minutes_hun_part8_202609272025.json` (937.0KB)
File saved to `senate_meeting_minutes\json\senate_meeting_minutes_hun_part9_202609272025.json` (757.5KB)
File saved to `senate_meeting_minutes\json\senate_mee